# Task 1.2 — Three Key Assumptions in MSVMpack

## Assumption 1: The Kernel is of Positive Type → Valid RKHS Exists

**Assumption:** The kernel $\kappa(x,x')$ is symmetric and positive semi-definite (Mercer condition), so it defines a valid RKHS $\mathcal{H}_\kappa$.

**Why the method needs it:** Section 2 represents each classifier component $h_k$ as an element of $\mathcal{H}_\kappa$, relying on the Representer Theorem to express $h_k$ as a finite kernel expansion. Without a PSD kernel the Gram matrix has negative eigenvalues, the QP in Definition 1 becomes non-convex, and the Wolfe dual may not equal the primal optimum — the entire Frank-Wolfe convergence analysis breaks down. MSVMpack explicitly lists only Mercer kernels (RBF, polynomial, linear) as valid inputs.

**Violation scenario:** A custom string-similarity measure that is not PSD on the training set produces an indefinite kernel matrix; the LP sub-problems in the Frank-Wolfe iterations receive unbounded feasible directions, causing divergence or convergence to a spurious local solution with arbitrarily poor test error.

**Paper reference:** Section 2 — definition of $\mathcal{H}_\kappa$ and the function class $\mathcal{H}$.

## Assumption 2: Convex QP with Valid Norm Matrix M → Unique Dual Solution

**Assumption:** The matrix $M$ is positive semi-definite on the relevant subspace, making the primal objective in Eq. 1 strictly convex and guaranteeing a unique primal-dual solution pair.

**Why the method needs it:** The duality-gap stopping criterion $U(\alpha)-D(\alpha)<\varepsilon$ (Section 3.1) is only meaningful when strong duality holds — i.e., when the primal minimum equals the dual maximum. Strict convexity (ensured by PSD $M$ and $\lambda>0$) is precisely what guarantees this via Slater's condition. If $M$ is invalid, the QP may be non-convex, the dual could be a strict under-estimate of the primal, and the solver would terminate with a certificate of optimality that is actually wrong.

**Violation scenario:** A user-constructed variant that accidentally sets $M$ to a matrix with a negative eigenvalue (e.g., a typo in the custom weight matrix) would silently break convexity; MSVMpack would still run and report a small duality gap, but the converged $\alpha$ would not correspond to the true global optimum, yielding a suboptimal classifier with no warning.

**Paper reference:** Definition 1 (Eq. 1); Table 1 — the four valid $(M,p,K_1,K_2,K_3)$ tuples are chosen specifically to maintain convexity.

## Assumption 3: Frank-Wolfe Working-Set Selection is Sufficient for Global Convergence

**Assumption:** Restricting each Frank-Wolfe LP to a small active subset (working set) of training examples still produces descent directions that eventually satisfy all KKT conditions globally, i.e., the working-set selection rule is rich enough to activate every binding constraint.

**Why the method needs it:** The decomposition strategy (Section 3.1) is what makes MSVMpack memory-feasible for large datasets like CB513. Its correctness rests on the guarantee that no persistently violated constraint is excluded from all working sets — otherwise the algorithm converges locally while a global violated constraint remains unaddressed. The convergence proof requires the selection heuristic to satisfy a sufficient-decrease property at every step.

**Violation scenario:** On a pathological dataset where many support vectors cluster in a region that the heuristic systematically deprioritises, the solver stagnates: the duality gap computed on the active set falls below $\varepsilon$ while the true global gap is still large. The returned model under-enforces margins for the ignored examples, showing higher-than-expected test error with no algorithmic warning.

**Paper reference:** Section 3.1 — decomposition method and the upper-bound stopping criterion $U(\alpha)$.